# Tracing Oliver's OODA loop via ReAct
### Secure Agentic AI Systems (COMP9098)

A standalone worked example, running the exact scenario from the slide you just saw: `"Oliver, check if Table 4's order contains allergens."` This notebook turns that diagram into real, running code, so you can watch the three boxes actually happen, line by line.

Nothing below is scripted or pre-written for you to read out. Oliver is a live LLM: every `THOUGHT:` and `ACTION:` line it prints is generated fresh, in its own words, the moment you run the cell.

## The three boxes on the slide, as code

| Slide box | What it means | Where you'll see it below |
| --- | --- | --- |
| **Step 1: Observe & Orient** | The raw fact ("Table 4 ordered Peanut Curry") gets appended to the conversation history the model can see. | The `messages` list, and the `OBSERVATION:` line printed each turn. |
| **Step 2: Decide** | The model reads that whole history and generates a `THOUGHT:` and an `ACTION:` line, token by token. Nothing is looked up here, it is a prediction. | The `THOUGHT:` / `ACTION:` text printed each turn, straight from `client.chat_completion(...)`. |
| **Step 3: Act (hard stop)** | Generation freezes the instant the model finishes the `ACTION:` line. Our own Python, not the model, reads that text and decides what to actually run. | The `ACTION_PATTERN` regex match, and the call into `TOOLS[...]`. |

Same engine as the `Observe -> Orient -> Decide -> Act` slide from earlier, just narrated the way the diagram just narrated it: two of the four phases (Observe, Orient) collapse into one "what does the model see" step, and Decide/Act are the two things that actually change the world.

## Setup, run once

In [ ]:
!pip install -q -U huggingface_hub
# ^ Installs/upgrades the Hugging Face client library inside this Colab
# environment. -q keeps the install output quiet; -U forces an upgrade even
# if an older version is already present, since a stale version is the most
# common cause of errors here.

import re
# re is Python's built-in regular-expressions module. It is not used in this
# cell; it is imported here because the loop further down uses it to pull
# the tool name and argument straight out of the model's own ACTION: line.

from huggingface_hub import InferenceClient, __version__ as hf_hub_version
# InferenceClient is the object we use to actually send messages to the
# model and get a reply back. __version__ is the library's own version
# string, renamed here to hf_hub_version purely so we can print it below.

print("huggingface_hub version:", hf_hub_version)
# Quick sanity check: if the upgrade above silently failed, this is the
# first place you would notice, since an old version is what causes the
# chat_completion calls further down to break.

HF_TOKEN = None
# Start with no token. Whichever of the two methods below succeeds fills
# this in; the rest of the notebook just uses this one variable.

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    # If this is actually running inside Google Colab, this reads the token
    # from Colab's built-in Secrets manager (the key icon in the left
    # sidebar), so the real token value never has to be typed into a cell.
except Exception:
    pass
    # Not running in Colab, or no secret named HF_TOKEN was set up there.
    # We deliberately swallow the error here rather than crashing, and fall
    # through to the manual prompt below instead.

if not HF_TOKEN:
    from getpass import getpass
    HF_TOKEN = getpass("Paste your Hugging Face token (hf_...): ")
    # Fallback: ask for the token directly. getpass hides the characters as
    # they are typed and never prints or saves the value anywhere in the
    # notebook's output, so it will not leak if this file is shared later.

MODEL_ID = "Qwen/Qwen2.5-Coder-32B-Instruct"
# The exact model Oliver runs on: Qwen2.5-Coder, 32 billion parameters,
# instruction-tuned. This is just a string, its path on the Hugging Face Hub.

client = InferenceClient(provider="auto", api_key=HF_TOKEN)
# Creates the actual connection object that every later call to the model
# goes through. provider="auto" lets Hugging Face route the request to
# whichever backend host currently serves this model, instead of us hard
# coding one that might be down or overloaded.

MAX_TURNS = 4
# Safety limit for the loop defined later: if Oliver never produces a
# FINAL ANSWER, the loop stops after this many Observe/Decide/Act turns
# instead of running forever.

print("Setup complete.")


## Oliver's tool for this scene

One argument in, one string out. This is the `query_allergen_db(item=...)` call named in the `ACTION:` line on the slide.

In [ ]:
ALLERGEN_REGISTER = {
    "peanut": "Contains peanuts. Registered allergen, do not serve without confirming with the guest.",
    "shellfish": "Contains shellfish. Registered allergen.",
    "gluten": "Contains gluten.",
}
# A plain Python dictionary standing in for "the restaurant's allergen
# database". Each key is an ingredient name; each value is the note Oliver
# should surface if a guest asks about it. This is the only "data source"
# in the whole notebook, everything else is either the model's own text or
# code we wrote.

def query_allergen_db(item: str) -> str:
    """Looks up whether an ingredient is in the restaurant's allergen register."""
    key = item.strip().lower()
    # Normalise whatever text the model sends us: strip stray spaces and
    # lowercase it, so "Peanut", "peanut ", and "PEANUT" all match the same
    # dictionary key. The model will not always format its argument the
    # same way twice, this line is what makes the lookup forgiving of that.
    if key in ALLERGEN_REGISTER:
        return ALLERGEN_REGISTER[key]
        # Known allergen: hand back the pre-written safety note.
    return f"'{item}' is not in the allergen register (not a known allergen for us)."
    # Not a registered allergen. This line, real Python code, is what
    # decides the answer, the model never sees or touches
    # ALLERGEN_REGISTER directly, it only ever sees what this function
    # returns.

TOOLS = {"query_allergen_db": query_allergen_db}
# A lookup table mapping the tool's name, exactly as the model will type it
# inside an ACTION: line (a plain string), to the actual Python function to
# call. This dictionary is what turns the text "query_allergen_db" into a
# real function call later in the loop.

print("Registered tools:", list(TOOLS.keys()))


## Oliver's system prompt for this scene

His identity, the one tool he may call, and the exact `THOUGHT:` / `ACTION:` / `FINAL ANSWER:` format. This is what makes his output land in exactly the three boxes on the slide, not because the model "knows" OODA, but because we told it to answer in this shape. Everything inside those lines, the actual reasoning, is the model's own.

In [ ]:
# Oliver's entire "brain" for this scene: his identity, the one tool he
# is allowed to call, and the exact THOUGHT: / ACTION: / FINAL ANSWER:
# format he must reply in. This is just a string, there is no logic to
# trace here, but it is the single most important piece of code in the
# notebook: every THOUGHT/ACTION the model produces below is shaped by
# these instructions, not by anything the model "knows" about OODA.
SYSTEM_PROMPT = """
You are Oliver, a secure waiter robot working the floor of a restaurant.

You may use this tool, EXACTLY as shown:
- query_allergen_db(item)

House rules:
- Never confirm a dish is safe without checking every ingredient the guest asked about against the allergen register.

Respond using EXACTLY this format, and nothing else:
THOUGHT: <your reasoning about what to do next, one or two sentences>
ACTION: <one tool call, e.g. query_allergen_db("peanut")>

Once the task is fully done, respond with:
THOUGHT: <final reasoning>
FINAL ANSWER: <a short message to the customer or manager>
""".strip()
# .strip() removes the one blank line the triple-quote block leaves at
# the very start and very end of the string, it does not touch anything
# in the middle.


## The loop itself

This is the OODA engine, with the box labels added as comments so today's trace and the slide match up exactly.

In [ ]:
ACTION_PATTERN = re.compile(r"ACTION:\s*(\w+)\((.*?)\)")
FINAL_PATTERN = re.compile(r"FINAL ANSWER:\s*(.*)")
# Both patterns are built once, here, outside the loop, so we are not
# recompiling the same regex on every turn. ACTION_PATTERN looks for the
# literal text "ACTION:" followed by something shaped like a function call,
# and captures two groups: group(1) is the function name (e.g.
# query_allergen_db), group(2) is whatever text sits inside the
# parentheses. FINAL_PATTERN looks for "FINAL ANSWER:" and captures
# everything after it as group(1).

def run_oliver(task: str, verbose: bool = True) -> str:
    # task: the plain-English instruction we hand Oliver for this run.
    # verbose: True prints the live trace as it happens (what we want in
    # class); set False to run silently and only get the final result.

    # STEP 1, Observe & Orient: the task and every later tool result join this
    # one list. This is the entire "input stream" the model ever sees.
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": task},
    ]
    # "system" is Oliver's standing instructions; "user" is the task for
    # this run. This list is the model's *only* memory, there is no other
    # state anywhere in this program, everything Oliver "knows" mid-task is
    # sitting in this one Python list.

    for turn in range(1, MAX_TURNS + 1):
        # One pass of this loop is exactly one Observe-Orient-Decide-Act
        # cycle. It runs at most MAX_TURNS times.
        if verbose:
            print(f"\n===== Turn {turn} =====")
            # Purely a visual divider in the printed output, for us to
            # follow along, it has no effect on Oliver's behaviour.

        # STEP 2, Decide: send the whole history so far and let the model
        # generate the next THOUGHT/ACTION text, live, right now.
        response = client.chat_completion(
            messages=messages, model=MODEL_ID, max_tokens=200
        ).choices[0].message.content
        # This line is the ONLY place the actual LLM is involved, everything
        # else in this notebook is plain Python. messages=messages sends the
        # full conversation so far; max_tokens=200 caps how much text the
        # model can generate in one turn so it cannot ramble forever;
        # .choices[0].message.content digs into the API's response object
        # to pull out just the plain text Oliver wrote (the API wraps it in
        # extra structure we do not need here).

        if verbose:
            print(response)
        messages.append({"role": "assistant", "content": response})
        # Appends Oliver's own output back into the conversation under the
        # "assistant" role. This is what makes it a real multi-turn
        # exchange with memory: next turn, the model will see its own
        # THOUGHT/ACTION from this turn as part of its input.

        final_match = FINAL_PATTERN.search(response)
        if final_match:
            return final_match.group(1).strip()
            # If this response contains a FINAL ANSWER line, the task is
            # done: pull out the text after "FINAL ANSWER:" (that is
            # group(1) from the regex), trim any stray whitespace, and
            # return it immediately, ending the function here.

        # STEP 3, Act, the hard stop: generation has already frozen after the
        # ACTION line. OUR Python, not the model, decides what to run and runs it.
        action_match = ACTION_PATTERN.search(response)
        if not action_match:
            observation = "No valid ACTION found. Remember the required format."
            # The model did not follow the required format at all. Rather
            # than crashing, we hand back a plain-text nudge, which becomes
            # the next thing Oliver reads, so he gets a chance to correct
            # himself on the next turn.
        else:
            tool_name, raw_arg = action_match.group(1), action_match.group(2).strip("'\" ")
            # group(1) is the function name the model typed (e.g.
            # "query_allergen_db"); group(2) is whatever was inside the
            # parentheses. .strip("'\" ") removes any stray quote marks or
            # spaces the model might have typed around its argument, e.g.
            # turning "peanut" or 'peanut' into plain peanut.
            observation = TOOLS[tool_name](raw_arg) if tool_name in TOOLS else f"Unknown tool '{tool_name}'."
            # This is the moment the action genuinely happens: TOOLS looks
            # up the real Python function by the name the model gave, and
            # calls it with the argument the model gave. If the model
            # invented a tool name we never registered, we return an error
            # string instead of letting the program crash.

        # Back to STEP 1, Observe: the tool's return value re-enters the input
        # stream as the next thing the model reads, and the loop restarts.
        if verbose:
            print(f"[OBSERVATION] {observation}")
        messages.append({"role": "user", "content": f"OBSERVATION: {observation}"})
        # Whatever just happened, the tool's answer or the format-error
        # nudge, is fed back into the conversation as a new "user" message,
        # labelled OBSERVATION so Oliver can clearly tell it apart from the
        # original task. This is the loop closing: the for-loop now moves
        # to the next turn, where Oliver reads this and generates a fresh
        # THOUGHT/ACTION.

    return "Oliver ran out of turns without finishing the task."
    # Only reached if the for-loop completes all MAX_TURNS without Oliver
    # ever producing a FINAL ANSWER, a safe fallback so the function always
    # returns something instead of returning nothing.


## Run the exact example from the slide

Run this and read the printed trace top to bottom. For every line, say out loud which of the three slide-boxes (Observe & Orient / Decide / Act) it belongs to before moving to the next line.

In [ ]:
result = run_oliver("Oliver, check if Table 4's order contains allergens. They ordered Peanut Curry.")
# Calls the whole loop defined above with this one task string, and runs it
# to completion (or until MAX_TURNS is hit). Everything prints live inside
# run_oliver itself, this line only kicks it off.
print("\n===== FINAL ANSWER =====")
print(result)
# result is whatever run_oliver returned, either the text after Oliver's own
# FINAL ANSWER: line, or the "ran out of turns" fallback string.


## Reading the trace back against the slide

Match what just printed to the three boxes:

- **`OBSERVATION: ...`, and the very first task line** -- Step 1, Observe & Orient. This is the raw input stream: the fact that Table 4 ordered Peanut Curry, and later, what the allergen register actually said. Nothing here was generated by the model, it is data going in.
- **`THOUGHT:` and `ACTION:`** -- Step 2, Decide. The model has read everything above and generated its reasoning and its next move, as plain text, one token at a time. It has not "looked up" the answer; it does not know yet whether peanut is registered, that is exactly why it calls the tool instead of guessing.
- **The moment between the `ACTION:` line and `[OBSERVATION] ...`** -- Step 3, Act, the hard stop. Token generation froze the instant the model finished that line. Our Python code, the `ACTION_PATTERN` regex and the `TOOLS[...]` lookup, is what actually ran `query_allergen_db`, not the LLM.
- **`FINAL ANSWER:`** -- the loop exiting once Oliver decides the task is done. Everything above it is exactly one Observe-Orient-Decide-Act cycle, repeated.

**Try it:** change the order to something not in the register, e.g. `"They ordered a plain rice dish."`, and re-run. Watch how the trace, and Oliver's own wording, changes, without one line of code changing.

In [ ]:
result = run_oliver("Oliver, check if Table 4's order contains allergens. They ordered a plain rice dish.")
# Same function, same code, only the task string changed, to a dish with no
# registered allergen. Nothing below this line was rewritten for this case,
# Oliver has to work it out himself from the tool's response.
print("\n===== FINAL ANSWER =====")
print(result)
